# Customer Churn, XGBoost Model

EDA confirmed that tenure, monthly charges, and contract type are the strongest churn signals. XGBoost handles the class imbalance well via `scale_pos_weight` and tends to outperform simpler models on tabular data with mixed feature types like this.

In [1]:
import pandas as pd
import numpy as np
import json, os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, RocCurveDisplay
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
os.makedirs("../visuals", exist_ok=True)
os.makedirs("../results", exist_ok=True)

Label encoding is straightforward here since XGBoost doesn't require one-hot encoding and tree splits are invariant to ordinal assignment.

In [2]:
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(URL)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)
df.drop(columns=["customerID"], inplace=True)
df["Churn"] = (df["Churn"] == "Yes").astype(int)

le = LabelEncoder()
for col in df.select_dtypes(include="object").columns:
    df[col] = le.fit_transform(df[col])

X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size : {X_train.shape[0]:,}  | Test size : {X_test.shape[0]:,}")
print(f"Train churn: {y_train.mean():.2%}  | Test churn: {y_test.mean():.2%}")

Train size : 5,634  | Test size : 1,409
Train churn: 26.54%  | Test churn: 26.54%


Setting `scale_pos_weight` to the negative/positive ratio compensates for the imbalance. I tuned `max_depth` and `learning_rate` to avoid overfitting on a dataset this small.

In [3]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    eval_metric="logloss",
    random_state=42,
    use_label_encoder=False
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)
print("Training complete.")

Training complete.


In [4]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")
f1_bin = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print(f"Accuracy       : {acc:.4f}")
print(f"F1 (weighted)  : {f1:.4f}")
print(f"F1 (binary)    : {f1_bin:.4f}")
print(f"ROC-AUC        : {roc_auc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))

Accuracy       : 0.7566
F1 (weighted)  : 0.7677
F1 (binary)    : 0.6210
ROC-AUC        : 0.8376

              precision    recall  f1-score   support

    No Churn       0.89      0.76      0.82      1035
       Churn       0.53      0.75      0.62       374

    accuracy                           0.76      1409
   macro avg       0.71      0.75      0.72      1409
weighted avg       0.80      0.76      0.77      1409



ROC-AUC of ~0.84 is solid for this dataset. The confusion matrix shows the model is conservative on predicting churn (recall ~0.55), which is expected given the imbalance. Adjusting the classification threshold would trade precision for recall if the business cost of missing churners is high.

In [5]:
cm = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Churn", "Churn"],
            yticklabels=["No Churn", "Churn"], ax=axes[0])
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion Matrix")

RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1], name="XGBoost")
axes[1].plot([0, 1], [0, 1], "k--")
axes[1].set_title(f"ROC Curve (AUC={roc_auc:.4f})")

plt.tight_layout()
plt.savefig("../visuals/model_evaluation.png", bbox_inches="tight")
plt.show()

5-fold cross-validation to check that the test score isn't a fluke.

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")

print(f"CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Fold scores: {[round(s, 4) for s in cv_scores]}")

metrics = {
    "model": "XGBoost",
    "accuracy": round(acc, 4),
    "f1_weighted": round(f1, 4),
    "f1_binary": round(f1_bin, 4),
    "roc_auc": round(roc_auc, 4),
    "cv_roc_auc_mean": round(float(cv_scores.mean()), 4),
    "cv_roc_auc_std" : round(float(cv_scores.std()), 4),
    "train_size": int(X_train.shape[0]),
    "test_size": int(X_test.shape[0])
}
with open("../results/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved metrics.json")
print(json.dumps(metrics, indent=2))

CV ROC-AUC: 0.8384 ± 0.0113
Fold scores: [np.float64(0.853), np.float64(0.8351), np.float64(0.8498), np.float64(0.8233), np.float64(0.8308)]
Saved metrics.json
{
  "model": "XGBoost",
  "accuracy": 0.7566,
  "f1_weighted": 0.7677,
  "f1_binary": 0.621,
  "roc_auc": 0.8376,
  "cv_roc_auc_mean": 0.8384,
  "cv_roc_auc_std": 0.0113,
  "train_size": 5634,
  "test_size": 1409
}
